In [3]:
df = pd.read_csv('../data/final_panel_data.csv')
df = df.loc[df['isocode'] != 'KOR'].copy()
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.set_index(['isocode', 'period']).sort_index()
print(df.shape)
df.head(3)

FileNotFoundError: [Errno 2] No such file or directory: '../data/final_panel_data.csv'

In [2]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller

# Define the 8 Macro Variables from your dataset
MACRO_VARS = ['brent_oil', 'inflation', 'fx_rate', 'ip', 'stock_index', 'terms_of_trade', 'reserves', 'debt']
TARGET_VAR = 'yield_10y' # Use the continuous yield for Granger tests

def ensure_stationarity(df, vars_list):
    """Checks ADF and returns a dataframe with stationary transformations."""
    df_stationary = df.copy()
    stationary_vars = []
    
    print("--- Stationarity Log (ADF Test) ---")
    for var in vars_list:
        # Drop NAs for the test; assumes panel data is sorted by date
        series = df[var].dropna()
        if len(series) < 20: continue
        
        result = adfuller(series)
        p_val = result[1]
        
        if p_val > 0.05:
            print(f"[DIFF] {var} is non-stationary (p={p_val:.4f}). Applying first difference.")
            # Apply difference within each country to avoid cross-country leakage
            df_stationary[var] = df.groupby('isocode')[var].diff()
        else:
            print(f"[OK] {var} is stationary (p={p_val:.4f}).")
        stationary_vars.append(var)
        
    return df_stationary.dropna(subset=vars_list)

# Execution
df_stat = ensure_stationarity(df, MACRO_VARS + [TARGET_VAR])

NameError: name 'df' is not defined